# 路由训练的时间外配置预测
固定年度2020/2021/2022，沿用原分折截止及300轮，三seed等权，不选模型。预测按实际交易年归属，禁止把2023+回测纳入路由训练。此卡只生成配置层OOF预测，不声称策略独立验证。

In [1]:
from pathlib import Path
from dataclasses import asdict
import os,json,hashlib
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from src.workflow.minute_action_value import _source
from src.workflow.minute_pit_allocator import _primary_dataset,_auxiliary_dataset,_prediction_dates,_ensemble
from src.model.tangent_allocator import TangentHeadAllocator,audit_tangent_initialization
from src.data.nas_market_data import archive_nas_recorder
from src.data.minute_intake import atomic_json
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/tangent_router_oof_20261015';plan=json.loads((card/'plan.json').read_text());provider=root/'data/qlib/full_path_tiered_allocation_20261013'
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(provider),region='us')
assert hashlib.sha256((root/'production/stage1_futu_sim_v1/runtime_release.json').read_bytes()).hexdigest()==plan['release_sha256']
names=['inputs/daily_frame','inputs/features','inputs/common','inputs/timing','inputs/auxiliary','inputs/audit','inputs/plan',*[f'models/base_{year}_{seed}' for year in plan['years'] for seed in plan['base_seeds']]]
source,audit=_source(root,plan['training_source'],names)
frame=source['inputs/daily_frame'];base_inputs={'frame':frame,'features':source['inputs/features'],'common':source['inputs/common'],'timing':source['inputs/timing']}
datasets={};auxiliary={};domains={}
for year in plan['years']:
 original=source[f'models/base_{year}_{plan["base_seeds"][0]}'];dates=frame.index[pd.to_datetime(frame.trade_date).dt.year.eq(year)]
 assert len(dates)>200 and pd.to_datetime(frame.loc[dates,'trade_date']).min().tz_localize('America/New_York')>original.cutoff
 domains[year]=dates
 datasets[year]=_primary_dataset({**base_inputs,'spec':original.spec},train_end=str(original.train_end.date()),predict_bounds=(str(dates.min().date()),str(dates.max().date())))
 auxiliary[year]=_auxiliary_dataset(source['inputs/auxiliary'][str(year)],str(original.train_end.date()))
progress_path=card/'run_state.json';digest=hashlib.sha256((card/'plan.json').read_bytes()).hexdigest();progress=json.loads(progress_path.read_text()) if progress_path.exists() else {'plan_sha256':digest,'fitted':[]}
assert progress['plan_sha256']==digest
kwargs={'experiment_name':card.name,'uri':str(card/'qlib_recorder')}
if progress.get('recorder_id'):kwargs.update(recorder_id=progress['recorder_id'],resume=True)
training=[];predictions={};folds=[]
with R.start(**kwargs):
 rec=R.get_recorder();progress.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(progress,progress_path)
 rec.save_objects(artifact_path='inputs',plan=plan,source_audit=audit);rec.save_objects(artifact_path='sources',model=(root/'src/model/tangent_allocator.py').read_text(),parent_model=(root/'src/model/minute_pit_allocator.py').read_text());rec.log_params(maximum_new_fits=9,new_backtests=0)
 for year in plan['years']:
  for seed in plan['base_seeds']:
   original=source[f'models/base_{year}_{seed}'];name=f'{year}_{seed}'
   assert original.spec.epochs==plan['epochs'] if hasattr(original.spec,'epochs') else original.audit['epochs']==plan['epochs']
   model=TangentHeadAllocator(base_features=original.base_features,extra_features=[],feature_masks=original.feature_masks,spec={**asdict(original.spec),'root':str(root)},seed=original.seed,cutoff=str(original.cutoff),train_end=str(original.train_end.date()),minimum_observations=original.minimum_observations)
   initialization=audit_tangent_initialization(original._initialize(),model._initialize())
   if name in progress['fitted']:model=rec.load_object('models/'+name)
   else:
    assert len(progress['fitted'])<plan['maximum_new_fits'];model.fit(datasets[year],auxiliary_dataset=auxiliary[year]);rec.save_objects(artifact_path='models',**{name:model});progress['fitted'].append(name);atomic_json(progress,progress_path)
   for key in ['training_rows','first_training_signal','last_training_signal','last_mature_label_at','auxiliary_rows','epochs','checkpoint_policy']:assert model.audit[key]==original.audit[key],key
   assert model.primary_scaler==original.primary_scaler and model.common_scaler==original.common_scaler
   pred=_prediction_dates(model.predict(datasets[year]));assert pred.index.equals(domains[year]);assert np.isfinite(pred.to_numpy(float)).all()
   predictions[(str(year),seed)]=pred;training.append({'year':year,'base_seed':seed,'actual_seed':model.seed,'prediction_rows':len(pred),'first_prediction_signal':str(pred.index.min().date()),'first_prediction_trade':str(frame.loc[pred.index,'trade_date'].min().date()),'last_prediction_trade':str(frame.loc[pred.index,'trade_date'].max().date()),**model.audit,'initialization':initialization});rec.log_metrics(completed_fits=len(progress['fitted']));print('fit',name,flush=True)
  ensemble=_ensemble(predictions,str(year),plan['base_seeds']);ensemble['model_year']=year;ensemble['trade_date']=frame.loc[ensemble.index,'trade_date'];folds.append(ensemble)
 oof=pd.concat(folds).sort_index();assert not oof.index.duplicated().any();assert pd.to_datetime(oof.trade_date).dt.year.eq(oof.model_year).all()
 assert (oof.index.intersection(frame.index[pd.to_datetime(frame.trade_date).dt.year.ge(2023)])).empty
 rec.save_objects(artifact_path='analysis',predictions=predictions,oof_predictions=oof,training=training);rec.log_metrics(oof_rows=len(oof))
t=pd.DataFrame([{k:v for k,v in row.items() if k not in ['initialization','observed_minute_training_rows','extra_scaler']} for row in training]);t.to_csv(card/'training_audit.csv',index=False)
policy={'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['models/*','analysis/predictions','analysis/oof_predictions','analysis/training','inputs/*','sources/*']}
summary={'experiment':card.name,'decision':'oof_inputs_ready_not_promoted','new_fits':len(progress['fitted']),'new_backtests':0,'oof_rows':len(oof),'first_signal':str(oof.index.min().date()),'last_signal':str(oof.index.max().date()),'provider_uri':str(provider),'region':'us','as_of':'2026-10-09','recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(card/'qlib_recorder'),'recorder_local_dir':str(card/'qlib_recorder'/str(rec.experiment_id)/rec.id),'artifact_policy':policy,'objective_achieved':False,'independent_strategy_validation':False,'scope':plan['scope']}
rec.save_objects(artifact_path='analysis',summary=summary);rec.log_metrics(registered_artifact_count=17)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'artifact_policy':policy}))
binding=json.loads((root/plan['training_source']['card']/'nas_binding.json').read_text());binding.pop('evidence',None);(card/'nas_binding.json').write_text(json.dumps(binding,indent=2))
a=Path(summary['recorder_local_dir'])/'artifacts';(card/'verification_manifest.json').write_text(json.dumps({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(a.rglob('*')) if p.is_file()]},indent=2))
summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));(card/'summary.json').write_text(json.dumps(summary,indent=2));print(t[['year','base_seed','training_rows','auxiliary_rows','prediction_rows','first_prediction_signal','last_mature_label_at']].to_string(index=False))


# 路由重训准备：9个小配置模型与756日时间外预测

本轮实际完成9次拟合，0新增账户回测。采用长期小配置头原39主输入、共同编码器、原日线收益及分类目标、300轮固定末轮、既定年度seed规则。2020/2021/2022各训练三个seed，年度内等权组合，不按收益选择模型。数据由已验证Qlib Recorder的冻结输入经DatasetH进入训练；未改变生产或原外层模型。

| 预测交易年 | 主训练截止 | 每模型主样本数 | 每模型辅助样本数 | 预测交易日数 |
|---|---|---:|---:|---:|
| 2020 | 2019-12-02 | 2244 | 0 | 253 |
| 2021 | 2020-12-02 | 2497 | 6578 | 252 |
| 2022 | 2021-12-02 | 2749 | 13007 | 251 |

各年最后主标签成熟时间分别为前一年12月10日09:31，均早于预测年1月1日训练cutoff；保留原分折隔离间隔，不扩充截止日。原年度模型的样本数、成熟时间、辅助样本数、训练轮数及缩放器逐项一致，小配置头初始化按既定局部切线合同核验。2020辅助样本为0，是冻结PIT覆盖的真实限制，未填补后期成员信息。

预测按实际交易年归属：第一条信号2019-12-31对应2020-01-02，最后信号2022-12-29对应2022-12-30；共756条，无重复，跨年信号不混用前一年模型。未生成或选择2023+预测。所有大预测矩阵、模型和完整训练审计存于Recorder；小汇总见training_audit.csv。

这些是配置层训练时序上的OOF预测，不是新的独立策略验证：小模型结构已在历史研究中被选择，研究者也已看过2023+结果，不能把时间切分自动解释为未受选择影响的证据。尚未构造执行/HOLD动作标签，尚未拟合新路由，尚未验证任何新增收益。

下一步固定构造同日起点、同成交价格与费用的执行/持有分支，标签成熟后才能训练。最后几行预测若其动作收益跨入2023，必须按既定cutoff排除训练资格，而不是因为预测年为2022就纳入。训练账户的状态轨迹也需因果合格：不能直接用训练至2022的冻结恢复/开盘模型在2020生成所谓时间外标签；应使用当时合格模型或明确固定、无学习的训练参考策略。跨年账户和目标平滑需连续，不重置以改善表现。该步骤仍待完成。

决策oof_inputs_ready_not_promoted。9次拟合和NAS冷恢复验证是研究准备进展，不是稳定净SOXX超额；目标未完成，生产未替换。不追加seed、epoch或样本窗搜索。
